# Analyse AAPL — préparation et première analyse

Ce notebook prépare et analyse un fichier historique de cours AAPL. Il ne passe aucun ordre et les données historiques ne constituent pas une recommandation d’investissement.

## 1. Préparer l’environnement sous Windows (PowerShell)

Exécutez ces commandes dans PowerShell **avant** d’ouvrir Jupyter. Elles créent `.venv` seulement s’il n’existe pas déjà, puis activent l’environnement et démarrent Jupyter. Le notebook doit être enregistré dans `Documents\AAPL_Intelligent_Trading`.

```powershell
$Project = "$HOME\Documents\AAPL_Intelligent_Trading"
New-Item -ItemType Directory -Force -Path $Project | Out-Null
Set-Location $Project
if (-not (Test-Path ".\.venv\Scripts\python.exe")) { py -3 -m venv .venv }
& ".\.venv\Scripts\Activate.ps1"
python -m pip install --upgrade pip
python -m pip install jupyter ipykernel yfinance pandas matplotlib
python -m jupyter notebook
```

Dans Jupyter, créez un notebook Python et enregistrez-le sous `analyse_aapl.ipynb` dans ce dossier. Les cellules ci-dessous s’exécutent dans le noyau sélectionné ; vérifiez que ce noyau utilise `.venv`.


## 2. Installer les dépendances (si nécessaire)

Cette cellule installe les bibliothèques dans le Python du noyau courant. Si vous venez de les installer dans `.venv` via PowerShell, vous pouvez la laisser telle quelle ou simplement vérifier les imports ensuite.


In [ ]:
%pip install yfinance pandas matplotlib


## 3. Télécharger les données et créer `AAPL.csv`

Le fichier sera écrit à côté du notebook. `auto_adjust=True` demande à yfinance des cours ajustés. Le service et les données peuvent être indisponibles ou changer de format.


In [ ]:
from pathlib import Path
import yfinance as yf

csv_path = Path('AAPL.csv')
data = yf.download('AAPL', period='1y', auto_adjust=True, progress=False)
print('Lignes téléchargées :', len(data))

if data.empty:
    raise RuntimeError('Aucune donnée reçue. Vérifiez votre connexion et réessayez plus tard.')

data.to_csv(csv_path)
print('CSV enregistré :', csv_path.resolve())


## 4. Charger le CSV et vérifier rapidement sa qualité

Contrôles simples : dimensions, colonnes, lignes dupliquées, cellules manquantes et cours de clôture. Les résultats doivent être examinés, pas seulement calculés.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(csv_path, header=[0, 1], index_col=0, parse_dates=True)

# yfinance peut produire des colonnes à deux niveaux; aplatir proprement.
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [str(col[0]) for col in df.columns]

print('Dimensions :', df.shape)
print('Colonnes :', df.columns.tolist())
print('Lignes dupliquées :', df.index.duplicated().sum())
print('Valeurs manquantes par colonne :\n', df.isna().sum())
display(df.head())


## 5. Tracer le cours de clôture ajusté


In [ ]:
close_columns = [column for column in df.columns if column == 'Close' or column.startswith('Close_')]
if not close_columns:
    raise ValueError(f'Colonne Close introuvable. Colonnes disponibles : {df.columns.tolist()}')

close = pd.to_numeric(df[close_columns[0]], errors='coerce').dropna()
if close.empty:
    raise ValueError('Aucun cours de clôture numérique exploitable.')
if (close <= 0).any():
    print('Avertissement : certains cours sont nuls ou négatifs.')

plt.figure(figsize=(11, 5))
plt.plot(close.index, close, linewidth=1.5)
plt.title('AAPL — cours de clôture ajusté (historique)')
plt.xlabel('Date')
plt.ylabel('Cours (USD)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Notes
- Ce notebook télécharge des données depuis Yahoo Finance via `yfinance` ; ce n’est pas une source garantie ni un flux temps réel.
- Les contrôles présentés sont élémentaires et ne garantissent pas l’exactitude des données.
- Le graphique décrit le passé. Il ne constitue pas un signal d’achat ou de vente.
